In [ ]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import sys
import os

# Add src to path
sys.path.append(os.path.abspath(os.path.join('..')))
from src import config, data_loader, features, evaluation, lstm_model

# Set plot style
plt.style.use('seaborn-v0_8')
print("Libraries loaded.")

In [ ]:
# Cell 2: Load and Prepare Data
# We need to reload data to get the exact Test set for comparison
print("Loading data...")
df = data_loader.download_data(config.TICKER, config.START_DATE, config.END_DATE)
df = features.add_technical_indicators(df)
data = df[['Close']]

# Split Data (Must match training split exactly)
train_size = int(len(data) * config.TRAIN_SPLIT)
train_data, test_data = data[:train_size], data[train_size:]
print(f"Test Data range: {test_data.index.min()} to {test_data.index.max()}")

In [ ]:
# Cell 3: Load Saved ARIMA Model
arima_path = os.path.join(config.MODELS_DIR, 'arima_model.pkl')
if os.path.exists(arima_path):
    print(f"Loading ARIMA model from {arima_path}...")
    arima_fit = joblib.load(arima_path)
    print("ARIMA model loaded successfully.")
else:
    print("Error: ARIMA model not found. Please run main.py or 02_arima_forecasting.ipynb first.")

In [ ]:
# Cell 4: Load Saved LSTM Model
lstm_path = os.path.join(config.MODELS_DIR, 'lstm_model.h5')
if os.path.exists(lstm_path):
    print(f"Loading LSTM model from {lstm_path}...")
    lstm_net = load_model(lstm_path)
    print("LSTM model loaded successfully.")
else:
    print("Error: LSTM model not found. Please run main.py or 03_lstm_forecasting.ipynb first.")

In [ ]:
# Cell 5: Generate ARIMA Predictions
# Get forecast for the test period
print("Generating ARIMA predictions...")
arima_forecast = arima_fit.get_forecast(steps=len(test_data))
arima_pred = arima_forecast.predicted_mean
arima_conf = arima_forecast.conf_int()

# Align index
arima_pred = pd.Series(arima_pred.values, index=test_data.index)

In [ ]:
# Cell 6: Generate LSTM Predictions
# Note: We need to re-create the scaler and sequences just like in training
print("Generating LSTM predictions...")
lstm_loader = lstm_model.LSTMForecaster(sequence_length=config.SEQ_LEN)
# We fit the scaler on ALL data (or just train) depending on how it was done in main.py. 
# In main.py, we passed the whole 'data' to preprocess, so it scaled based on the whole history.
# We will do the same here to ensure consistency.
X, y, _ = lstm_loader.preprocess(data)

# Calculate where the test set starts in the sequence data
# The sequences start from index 'seq_len'. 
# We need the sequences that correspond to the test_data indices.
split_idx = int(len(X) * config.TRAIN_SPLIT)
X_test = X[split_idx:]

# Predict
lstm_raw_pred = lstm_net.predict(X_test)
lstm_pred = lstm_loader.scaler.inverse_transform(lstm_raw_pred)

# Align LSTM predictions with Test Data
# Note: X_test might be slightly smaller than test_data due to sequence windowing at the boundary
# We slice test_data to match the length of LSTM predictions for comparison
test_data_lstm = test_data.iloc[-len(lstm_pred):]
lstm_pred_series = pd.Series(lstm_pred.flatten(), index=test_data_lstm.index)

In [ ]:
# Cell 7: Calculate Evaluation Metrics
print("\n--- Model Evaluation ---")
metrics_df = pd.DataFrame(index=['MAE', 'RMSE', 'MAPE', 'R2'])

# ARIMA Metrics
arima_metrics = evaluation.calculate_metrics(test_data['Close'], arima_pred, "ARIMA")
metrics_df['ARIMA'] = arima_metrics.values()

# LSTM Metrics
lstm_metrics = evaluation.calculate_metrics(test_data_lstm['Close'], lstm_pred_series, "LSTM")
metrics_df['LSTM'] = lstm_metrics.values()

print("\nComparison Table:")
display(metrics_df)

In [ ]:
# Cell 8: Final Comparison Plot
plt.figure(figsize=(15, 8))

# Plot Actual
plt.plot(test_data.index, test_data['Close'], label='Actual Price', color='black', linewidth=2)

# Plot ARIMA
plt.plot(test_data.index, arima_pred, label='ARIMA Forecast', color='blue', linestyle='--')

# Plot LSTM
plt.plot(test_data_lstm.index, lstm_pred_series, label='LSTM Forecast', color='green', linestyle='--')

plt.title(f'Final Comparison: {config.TICKER} Stock Price Forecasting')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(config.REPORTS_DIR, 'final_model_comparison.png'))
plt.show()

In [ ]:
# Cell 9: Error Distribution Analysis
# Calculate residuals
arima_resid = test_data['Close'] - arima_pred
lstm_resid = test_data_lstm['Close'] - lstm_pred_series

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(arima_resid, kde=True, color='blue')
plt.title('ARIMA Residuals Distribution')

plt.subplot(1, 2, 2)
sns.histplot(lstm_resid, kde=True, color='green')
plt.title('LSTM Residuals Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Conclusion
best_model = metrics_df.loc['RMSE'].idxmin()
print(f"Based on RMSE, the best performing model is: {best_model}")